In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0 databricks-feature-store==0.17.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.1/569.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 36.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.3/212.3 kB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 90.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 25.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 25.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 30.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 42.0 MB/s eta 0:00:00
     ━━━

In [0]:
%sql
select 
  is_fraud,
  count(1) as `Transactions`, 
  sum(amount) as `Total Amount` 
from asscom1fraudanalysis.bdcom1asses.gold_transactions
group by is_fraud


is_fraud,Transactions,Total Amount
true,104841,3.601917980568004E10
false,3228993,3.6231696761069556E11


In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = spark.sql(
    'select type, is_fraud, count(1) as count from asscom1fraudanalysis.bdcom1asses.gold_transactions group by type, is_fraud'
).toPandas()

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])
fig.add_trace(go.Pie(labels=df[df['is_fraud']]['type'], values=df[df['is_fraud']]['count'], title="Fraud Transactions", hole=0.6), 1, 1)
fig.add_trace(go.Pie(labels=df[~df['is_fraud']]['type'], values=df[~df['is_fraud']]['count'], title="Normal Transactions", hole=0.6), 1, 2)

fig.show()

In [0]:
# Convert to koalas
dataset = spark.table('asscom1fraudanalysis.bdcom1asses.gold_transactions').dropDuplicates(['id']).pandas_api()
# Drop columns we don't want to use in our model
# Typical DS project would include more transformations / cleanup here
dataset = dataset.drop(columns=['address', 'email', 'firstname', 'lastname', 'creation_date', 'last_activity_date', 'customer_id'])

# Drop missing values
dataset.dropna()
dataset.describe()

,amount,isUnauthorizedOverdraft,newBalanceDest,newBalanceOrig,oldBalanceDest,oldBalanceOrig,step,diffOrig,diffDest,age_group
count,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06,3.333834e+06
mean,1.194829e+05,2.294055e-03,9.965282e+05,2.763080e+06,9.622953e+05,2.773257e+06,2.966545e+02,-1.017711e+04,3.423291e+04,4.999508e+00
std,1.174984e+05,4.784134e-02,8.334800e+05,1.324057e+06,8.077723e+05,1.314829e+06,1.696047e+02,1.623762e+05,1.025743e+05,2.915154e+00
min,0.000000e+00,0.000000e+00,-1.686547e+04,-1.843748e+05,-2.479268e+04,-1.843748e+05,0.000000e+00,-4.998869e+05,0.000000e+00,0.000000e+00
25%,1.139945e+04,0.000000e+00,4.000000e+05,2.045789e+06,3.744138e+05,2.048540e+06,1.660000e+02,-7.750349e+04,0.000000e+00,3.000000e+00
50%,9.593195e+04,0.000000e+00,8.752558e+05,2.915263e+06,8.556236e+05,2.918142e+06,2.880000e+02,-3.358200e+03,0.000000e+00,5.000000e+00
75%,1.769101e+05,0.000000e+00,1.445871e+06,3.470119e+06,1.428661e+06,3.474129e+06,4.390000e+02,1.007145e+05,5.036420e+03,7.000000e+00
max,7.483238e+05,1.000000e+00,2.018745e+07,2.828175e+07,1.999966e+07,2.827540e+07,6.010000e+02,7.483238e+05,4.000000e+05,1.000000e+01


In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

try:
  #drop table if exists
  fs.drop_table('asscom1fraudanalysis.bdcom1asses.transactions_features')
except:
  pass

fs.create_table(
  name='asscom1fraudanalysis.bdcom1asses.transactions_features',
  primary_keys='id',
  schema=dataset.spark.schema(),
  description='These features are derived from the gold_transactions table in the lakehouse. created dummy variables for the categorical columns, cleaned up their names, and added a boolean flag for whether the transaction is a fraud or not.  No aggregations were performed.')

fs.write_table(df=dataset.to_spark(), name='asscom1fraudanalysis.bdcom1asses.transactions_features', mode='overwrite')
features = fs.read_table('asscom1fraudanalysis.bdcom1asses.transactions_features')
display(features)

2025/05/04 17:03:34 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['id'] of table 'asscom1fraudanalysis.bdcom1asses.transactions_features' to NOT NULL.
2025/05/04 17:03:43 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['id'] on table 'asscom1fraudanalysis.bdcom1asses.transactions_features'.
2025/05/04 17:03:45 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'asscom1fraudanalysis.bdcom1asses.transactions_features'.


id,amount,isUnauthorizedOverdraft,nameDest,nameOrig,newBalanceDest,newBalanceOrig,oldBalanceDest,oldBalanceOrig,step,type,diffOrig,diffDest,country,last_country_logged,age_group,is_fraud,countryOrig,countryOrig_name,countryLongOrig_long,countryLatOrig_lat,countryDest,countryDest_name,countryLongDest_long,countryLatDest_lat
000017bb-e4ce-4bea-8e80-7fcfb3286a54,150710.55,0,M2324176787,C8182674483,768866.16,4199853.57,768866.16,4350564.12,282,CASH_OUT,-150710.5499999998,0.0,COG,COG,8.0,false,BRA,Brazil,-55,-10,TUR,Turkey,35,39
000084ae-19bd-4574-964f-3f281a6e6e60,6581.93,0,M0622973396,C5938303102,2025278.71,3146939.78,2018696.79,3153521.7,594,PAYMENT,-6581.920000000391,6581.9199999999255,NOR,NOR,2.0,false,RUS,Russian Federation,100,60,PER,Peru,-76,-10
0000e0a6-bcaa-451c-a842-c46ef75cd951,400000.0,0,CC0683493538,C3653596535,400000.0,4105776.17,0.0,4505776.17,185,TRANSFER,-400000.0,400000.0,GIN,GIN,2.0,false,QAT,Qatar,51.25,25.5,BRA,Brazil,-55,-10
0001c3f5-d703-4629-a557-270fe10c9424,4594.92,0,M7158211193,C2313073353,662830.76,3877473.5,658235.84,3882068.42,231,PAYMENT,-4594.9199999999255,4594.920000000042,DJI,DJI,5.0,false,RUS,Russian Federation,100,60,REU,Réunion,55.6,-21.1
0001e5c1-370e-4015-822b-c711fb7eea72,17934.96,0,M9060417517,C6887180642,823982.24,4092490.32,806047.28,4110425.28,281,PAYMENT,-17934.959999999963,17934.959999999963,LAO,LAO,1.0,false,RUS,Russian Federation,100,60,PAN,Panama,-80,9
0001ee3d-1db2-412a-ae4d-4d6df729feb2,52306.44,0,M1172763078,C7661571635,815048.23,4432733.39,815048.23,4380426.95,285,CASH_IN,52306.43999999948,0.0,MCO,MCO,4.0,false,KHM,Cambodia,105,13,KHM,Cambodia,105,13
0003525b-cc2b-48ba-bbc7-b80d554998bc,115446.44,0,M3447318396,C4381143905,1662958.43,422584.84,1662958.43,307138.4,597,CASH_IN,115446.44,0.0,MMR,MMR,3.0,false,TUR,Turkey,35,39,CAN,Canada,-95,60
00037566-408c-44e2-8da1-0608293a5062,175775.2,0,M5633899972,C6610448555,490742.21,2266388.92,490742.21,2442164.13,183,CASH_OUT,-175775.20999999996,0.0,TGO,TGO,9.0,false,FRA,France,2,46,QAT,Qatar,51.25,25.5
0003c179-97f4-4c5b-8ddd-fb82545cd6e8,109485.45,0,M2960431439,C1477670218,966826.41,2712034.22,966826.41,2602548.76,289,CASH_IN,109485.46000000043,0.0,LVA,LVA,5.0,false,NGA,Nigeria,8,10,QAT,Qatar,51.25,25.5
0003ccb7-8da8-4265-84a3-61c2b3cf4261,49344.71,0,CC2573718548,C5535976791,938736.38,2946677.74,889391.67,2996022.45,420,TRANSFER,-49344.70999999996,49344.70999999996,BDI,BDI,3.0,false,FRA,France,2,46,CAN,Canada,-95,60


In [0]:
%pip install databricks-automl-runtime

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()
features = fs.read_table('asscom1fraudanalysis.bdcom1asses.transactions_features')

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()
features = fs.read_table('asscom1fraudanalysis.bdcom1asses.transactions_features')

# Random 25% sample
quarter_sample = features.sample(withReplacement=False, fraction=0.002, seed=42)

print(f"Original count: {features.count()}")
print(f"25% sample count: {quarter_sample.count()}")

Original count: 3333834
25% sample count: 6785


In [0]:
features = quarter_sample

In [0]:
from databricks import automl
# from databricks.automl import classify
from datetime import datetime

# Define your experiment path (customize this!)
xp_path = "/Shared/experiments/my_fraud_project_shifa"  # Or your user directory
xp_name = f"automl_fraud_{datetime.now().strftime('%Y-%m-%d_%H:%M:%S')}"  # Unique name

# Run AutoML
automl_run = automl.classify(
    experiment_name=xp_name,
    experiment_dir=xp_path,
    dataset=features,  # Your feature DataFrame (no sampling needed if not a demo)
    target_col="is_fraud",  # Your target column
    timeout_minutes=30  # Adjust as needed
)



2025/05/04 19:38:06 INFO databricks.automl.client.manager: AutoML will optimize for F1 score metric, which is tracked as val_f1_score in the MLflow experiment.
2025/05/04 19:38:07 INFO databricks.automl.client.manager: MLflow Experiment ID: 2311129494619525
2025/05/04 19:38:07 INFO databricks.automl.client.manager: MLflow Experiment: https://dbc-f450058a-abb8.cloud.databricks.com/?o=2459231659607272#mlflow/experiments/2311129494619525
2025/05/04 19:40:04 INFO databricks.automl.client.manager: Data exploration notebook: https://dbc-f450058a-abb8.cloud.databricks.com/?o=2459231659607272#notebook/2311129494619543
2025/05/04 20:09:05 INFO databricks.automl.client.manager: AutoML experiment completed successfully.


,Train,Validation,Test
f1_score,0.878,0.810,0.709
false_negatives,24.000,9.000,16.000
score,0.990,0.989,0.983
example_count,2831.000,1315.000,1375.000
recall_score,0.802,0.780,0.636
true_negatives,2707.000,1268.000,1324.000
false_positives,3.000,6.000,7.000
true_positives,97.000,32.000,28.000
accuracy_score,0.990,0.989,0.983
precision_recall_auc,0.956,0.873,0.821


In [0]:
# Get the best trial's MLflow run ID
best_run_id = automl_run.best_trial.mlflow_run_id

# Load the best model as a PySpark/Python UDF
import mlflow
best_model = mlflow.pyfunc.spark_udf(
    spark, 
    model_uri=f"runs:/{best_run_id}/model"
)

# Display metrics
display(automl_run.best_trial)

In [0]:
# Or show in notebook:
display(automl_run)

Journey to register the model in the catalog

In [0]:
# Step 0: Set MLflow to Unity Catalog *FIRST*
import mlflow
mlflow.set_registry_uri('databricks-uc')  # Important to set this first!

In [0]:
# Step 2: Register the best model into Unity Catalog
best_trial = automl_run.best_trial
best_model_path = best_trial.model_path

catalog = "asscom1fraudanalysis"    
db = "bdcom1asses"         
registered_model_name = "fraud_detection_shifa"  # Keep this name

# Full model name includes catalog and schema/database
full_model_name = f"{catalog}.{db}.{registered_model_name}"

model_registered = mlflow.register_model(
    model_uri=best_model_path,
    name=full_model_name  # Directly register into UC
)



Registered model 'asscom1fraudanalysis.bdcom1asses.fraud_detection_shifa' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/6 [00:00<?, ?it/s]

Created version '3' of model 'asscom1fraudanalysis.bdcom1asses.fraud_detection_shifa'.


Making Predictions

In [0]:
from databricks.feature_store import FeatureStoreClient
import pyspark.sql.functions as F
import mlflow
import pandas as pd
# 1. Load 10 random samples
fs = FeatureStoreClient()
features = fs.read_table('asscom1fraudanalysis.bdcom1asses.transactions_features')
random_10_samples = features.orderBy(F.rand(seed=42)).limit(10).toPandas()
# 2. Load model and get its schema
model_uri = f"models:/{full_model_name}/{model_registered.version}"
model = mlflow.pyfunc.load_model(model_uri)
input_schema = model.metadata.get_input_schema()
# 3. Schema Alignment
required_columns = input_schema.input_names()
required_dtypes = {field.name: field.type for field in input_schema.inputs}
# Map model schema types to pandas types
dtype_mapping = {'integer': 'int64','long': 'int64','float': 'float64','double': 'float64','string': 'object','boolean': 'bool'}
# Create DataFrame with correct columns and dtypes
model_ready_samples = pd.DataFrame(columns=required_columns)
# Populate with available data (maintaining order and types)
for col in required_columns:
    if col in random_10_samples.columns:
        pandas_dtype = dtype_mapping.get(required_dtypes[col], 'object')
        model_ready_samples[col] = random_10_samples[col].astype(pandas_dtype)
    else:
        # Handle missing columns (fill with defaults or raise error)
        print(f"Warning: Missing required column {col}")
        model_ready_samples[col] = 0  # Or appropriate default
# 4. Get predictions
predictions = model.predict(model_ready_samples)
# 5. Display results
results = model_ready_samples.copy()
results['prediction'] = predictions
display(results)

id,amount,isUnauthorizedOverdraft,nameDest,nameOrig,newBalanceDest,newBalanceOrig,oldBalanceDest,oldBalanceOrig,step,type,diffOrig,diffDest,country,last_country_logged,age_group,countryOrig,countryOrig_name,countryLongOrig_long,countryLatOrig_lat,countryDest,countryDest_name,countryLongDest_long,countryLatDest_lat,prediction
119fcd6c-e3b7-470a-aed4-cea19ca19e32,75244.61,0,M3564306529,C0630678764,1204766.03,2565237.57,1204766.03,2489992.96,358,CASH_IN,75244.60999999987,0.0,KHM,KHM,3.0,PER,Peru,-76,-10,CAN,Canada,-95,60,false
fbf27a9d-0ce4-4ed8-8ab7-e3cdb2b0f5f8,400000.0,0,C6696998001,C2714682212,3847373.43,2129497.99,3847373.43,2129497.99,262,TRANSFER,0.0,0.0,PAN,PAN,10.0,KHM,Cambodia,105,13,RUS,Russian Federation,100,60,false
7e77ea30-802a-4eaa-be1a-9967402b7d78,1704.01,0,M6107761095,C6163823071,1299253.22,3428914.78,1297549.2,3430618.79,413,PAYMENT,-1704.0100000002421,1704.0200000000186,MLT,MLT,2.0,RUS,Russian Federation,100,60,FRA,France,2,46,false
d150cf70-d2fb-4c72-8868-9f251be0437f,226118.33,0,M3922112927,C0903798982,1417292.1,3403035.14,1417292.1,3176916.82,471,CASH_IN,226118.3200000003,0.0,MOZ,MOZ,6.0,PAN,Panama,-80,9,TUR,Turkey,35,39,false
d4edc024-9f79-485f-9599-bc7eecfca385,7510.3,0,M7356661228,C7144849193,327805.45,2067430.49,320295.15,2074940.79,138,PAYMENT,-7510.300000000047,7510.299999999988,POL,POL,5.0,RUS,Russian Federation,100,60,ESP,Spain,-4,40,false
6276b396-37c3-466c-98a1-534806560c7c,12417.45,0,M0661923216,C1295937017,929224.92,1044331.14,916807.48,1056748.59,285,PAYMENT,-12417.45000000007,12417.44000000006,GAB,GAB,1.0,BRA,Brazil,-55,-10,PSE,"Palestinian Territory, Occupied",35.25,32,false
466b2d1a-0e09-4f1e-8cc1-6245ca05f2e1,260537.57,0,M2255448446,C9561184195,1075309.53,4161753.61,1075309.53,3901216.04,372,CASH_IN,260537.56999999983,0.0,LBR,LBR,1.0,RUS,Russian Federation,100,60,KHM,Cambodia,105,13,false
88f85c53-b776-47c9-89dc-830bd03d891d,71910.59,0,M7603826496,C6770133307,1051700.95,3754684.06,1051700.95,3682773.47,267,CASH_IN,71910.58999999985,0.0,ESP,ESP,1.0,RUS,Russian Federation,100,60,ITA,Italy,12.8333,42.8333,false
1ef9c86f-fb37-490c-9cc3-667932ed4b72,349751.05,0,M5710401947,C2943099400,1029364.66,1266094.79,1029364.66,916343.74,325,CASH_IN,349751.05000000005,0.0,GNB,GNB,6.0,KHM,Cambodia,105,13,NGA,Nigeria,8,10,false
426fe749-b6dd-4ece-907c-41653e9eab6e,260820.86,0,M7600884913,C1110097094,0.0,2790270.99,0.0,2529450.13,8,CASH_IN,260820.86000000034,0.0,NRU,NRU,1.0,ITA,Italy,12.8333,42.8333,CAN,Canada,-95,60,false
